In [1]:
import numpy as np
from scipy.io import FortranFile

import os
import shutil

In [2]:
def name_formatter(p, specie, x, t):
    return f"spectrum/{p:04.1f}/{specie}/{x:.2f}_{int(t):04d}.npy"

In [3]:
for vf in [0.0, 0.25, 0.5, 0.75, 1.0]:
    for specie in ["H2O", "CO2", "CO_"]:
        db_name = f"test_db/absc{specie}.28.P.01.0m.{vf:.2f}.sp.vt.dat"
        # specie = "H2O"


        with FortranFile(db_name) as file:
            pressure, vf = file.read_reals(dtype="float32") # pressure, volume fraction
            
            base_dir = f"spectrum/{pressure:04.1f}/{specie}"

            try:
                os.makedirs(base_dir)
            except FileExistsError:
                pass
            wn_min, wn_max = file.read_reals(dtype="float32")
            n_temperature = file.read_reals(dtype=np.int32)[0]
            temperatures = np.zeros(n_temperature)

            for i in range(n_temperature):
                # Temperature, d_wn
                temperatures[i], d_wn = file.read_reals(dtype="float32")
                n_wn = file.read_reals(dtype=np.int32)[0]
                read_kappas = file.read_reals(dtype="float32")
                file_name = name_formatter(pressure, specie, vf, temperatures[i])
                np.save(file_name, read_kappas)